# 16 — HTTP, JSON APIs, and Reliable Clients

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/16', {1: 'd7e7ee4e3014dab8', 2: '0c36e8a1b939e979', 3: 'edf346e2cb58742b', 4: 'f09bc8fa2b1605e8', 5: '5db6087400dd3c99', 6: '2767a9c7d3926a0d', 7: '07b5a5dc85cb8dc7', 8: '83c921ffabf1eefd', 9: '31b208ec9776ada7', 10: '03ca60d95ef0e4ed'}, practice='intermediate_04_sql_http_cli')

## 1. Requests and URLs

- GET reads data; POST creates; PUT/PATCH update; DELETE removes. Status 2xx success, 4xx your mistake, 5xx the server's.
- `urllib.request.Request(url, headers={...})` builds a request; read headers back with `req.get_header("Accept")`.
- `urllib.parse.urlencode({"q": "a b"})` encodes a query string: `q=a+b`.

In [ ]:
import urllib.request, urllib.parse
req = urllib.request.Request("https://api.example.com/items", headers={"Accept": "application/json", "User-Agent": "course-client/1.0"})
print(req.get_method(), req.full_url, req.get_header("Accept"))
print("https://api.example.com/search?" + urllib.parse.urlencode({"q": "a b", "page": 2}))

**Exercise 1.** Which status range means the client sent a bad request? Write `"2xx"`, `"4xx"` or `"5xx"`.

In [ ]:
answer = ___
check(1, answer)

**Exercise 2.** Build a request with an `Accept: application/json` header.

In [ ]:
import urllib.request
req = urllib.request.Request("https://x.test/a", headers={"Accept": ___})
check(2, req.get_header("Accept"))

**Exercise 3.** Encode the query string.

In [ ]:
import urllib.parse
query = urllib.parse.urlencode(___)
check(3, query)   # should be "q=red+shoes&page=1"

**Exercise 4.** Join base and path with exactly one `/`.

In [ ]:
base, path = "https://api.x/", "/users"
url = base.rstrip("/") + "/" + path.lstrip("" + ___)
check(4, url)

## 2. Fake responses, real JSON

- A response body is bytes: `json.loads(response.read())` decodes it.
- Inject the opener (`opener=urllib.request.urlopen`) so tests pass a fake and never touch the network.
- Always pass a `timeout`. Turn bad JSON into a clear `ValueError`.

In [ ]:
import io, json

def fake_opener(request, timeout):
    return io.BytesIO(b'{"id": 7, "name": "Ana"}')

def fetch_json(url, opener, timeout=5.0):
    with opener(url, timeout=timeout) as response:
        body = response.read()
    try:
        return json.loads(body)
    except json.JSONDecodeError as error:
        raise ValueError("response was not valid JSON") from error

print(fetch_json("https://api.example.com/me", fake_opener))

**Exercise 5.** `json.loads` decodes the bytes body. Predict the result (JSON `true` is `True`).

In [ ]:
import io, json
response = io.BytesIO(b'{"ok": true}')
data = json.loads(response.read())
predicted = ___   # what data is, as a Python value
check(5, predicted)

**Exercise 6.** What type does `response.read()` return? Write its name.

In [ ]:
import io
predicted = ___   # type(io.BytesIO(b"{}").read()).__name__
check(6, predicted)

**Exercise 7.** A fake opener records the timeout it was given. What was it?

In [ ]:
import io
seen = {}
def fake_opener(request, timeout):
    seen["timeout"] = timeout
    return io.BytesIO(b"[]")

with fake_opener("u", timeout=2.5) as r:
    r.read()
predicted = ___
check(7, predicted)

## 3. Pagination

- APIs return pages: `{"items": [...], "next": "page2"}`. Keep fetching until `next` is `None`.
- Guard against loops with a `max_pages` limit.
- Offset pages: page `p` of size `n` starts at `(p - 1) * n`; total pages is `ceil(len / n)`.

In [ ]:
pages = {"p1": {"items": [1, 2], "next": "p2"}, "p2": {"items": [3], "next": None}}
items, page = [], "p1"
while page:
    data = pages[page]
    items.extend(data["items"])
    page = data["next"]
print(items)
import math
rows = list(range(1, 11))
print(rows[(2 - 1) * 4: 2 * 4], math.ceil(len(rows) / 4))

**Exercise 8.** Follow the pages and collect every item.

In [ ]:
pages = {"a": {"items": ["x"], "next": "b"}, "b": {"items": ["y", "z"], "next": None}}
items, page = [], "a"
while page:
    items.extend(pages[page]["items"])
    page = pages[page][___]
check(8, items)

**Exercise 9.** Page 3 of size 2 from `range(1, 8)`.

In [ ]:
items = list(range(1, 8))
page, per_page = 3, 2
start = (page - 1) * per_page
result = items[start:___]
check(9, result)

**Exercise 10.** How many pages for 7 items at 3 per page? And for 0 items?

In [ ]:
import math
predicted = ___
check(10, predicted)

## Challenges

Real functions, graded by tests. Fill every `___`, run the function cell, then run its check.

### 1. `build_request`

A GET request with JSON `Accept` and a `User-Agent`.

In [ ]:
def build_request(url: str) -> urllib.request.Request:
    """Build a GET Request with headers Accept: application/json and User-Agent: course-client/1.0."""
    return ___

In [ ]:
check('build_request')

### 2. `fetch_json`

Fetch and decode JSON through an injected opener.

In [ ]:
def fetch_json(url: str, opener: Callable = urllib.request.urlopen, timeout: float = 5.0):
    """Fetch and decode JSON. `opener(request, timeout=...)` returns an object with .read().

    Raise ValueError if the body is not valid JSON. The opener is injected so
    tests never touch the network.
    """
    with opener(build_request(url), timeout=timeout) as response:
        body = ___
    try:
        return ___
    except json.JSONDecodeError as error:
        raise ValueError("response was not valid JSON") from error

In [ ]:
check('fetch_json')

### 3. `collect_pages`

Follow `next` links; raise `RuntimeError` past `max_pages`.

In [ ]:
def collect_pages(fetch_page: Callable[[str], dict], first: str, max_pages: int = 100) -> list:
    """Follow {'items': [...], 'next': <page or None>} pages and return all items.

    Raise RuntimeError if more than max_pages pages are visited (protects against loops).
    """
    items, page, visited = ___
    while ___:
        visited += 1
        if ___:
            raise RuntimeError("too many pages")
        data = fetch_page(page)
        items.extend(data["items"])
        page = data.get("next")
    return items

In [ ]:
check('collect_pages')

### 4. `build_url`

Join base and path, then add an encoded query string.

In [ ]:
def build_url(base: str, path: str, params: dict | None = None) -> str:
    """Join base and path with exactly one '/', then add an encoded query string.

    build_url('https://api.x/', '/search', {'q': 'a b'}) -> 'https://api.x/search?q=a+b'
    Hint: urllib.parse.urlencode
    """
    import urllib.parse
    url = base.rstrip("/") + "/" + path.lstrip("/")
    return ___

In [ ]:
check('build_url')

### 5. `paginate`

One page of items plus the page count; reject pages below 1.

In [ ]:
def paginate(items: list, page: int, per_page: int) -> dict:
    """Return {'items': page slice, 'page': page, 'pages': total pages}. Pages start at 1.
    Raise ValueError for page < 1 or per_page < 1. An empty list has 0 pages."""
    if ___:
        raise ValueError("page and per_page must be at least 1")
    start = (page - 1) * per_page
    return ___

In [ ]:
check('paginate')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Why always set a timeout on HTTP calls?

   - **a)** To make requests faster
   - **b)** Servers require it
   - **c)** So a slow server can't hang your program forever

**2.** Which method should be safe to repeat with no side effects?

   - **a)** GET
   - **b)** Both equally
   - **c)** POST

**3.** Injecting the opener/transport into fetch functions lets you…

   - **a)** Skip JSON parsing
   - **b)** Use less memory
   - **c)** Test without the network

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/16', {1: '9697e9f328d9', 2: '641aae1308f5', 3: 'bb4f4680578a'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [17 — Command-Line Interfaces and Project Packaging](17_cli_and_packaging.ipynb)